In [0]:
from pyspark.sql import functions as F

sales_hourly = spark.read.table("sales_project_streaming.gld.hourly_regional_sales_agg")
exp_hourly = spark.read.table("sales_project_streaming.gld.hourly_regional_expenses_agg")

combined = (sales_hourly.alias("s").join(exp_hourly.alias("e"), on = ["window_start", "city"], how = "full_outer")
            .select(F.coalesce(F.col("s.window_start"), F.col("e.window_start")).alias("window_start"),
                    F.coalesce(F.col("s.window_end"), F.col("e.window_end")).alias("window_end"),
                    F.coalesce(F.col("s.city"), F.col("e.city")).alias("city"),
                    F.coalesce(F.col("s.total_revenue"), F.lit(0)).alias("total_revenue"),
                    F.coalesce(F.col("e.total_spent"), F.lit(0)).alias("total_spent"))
            )

display(combined)

In [0]:
%sql
select * from sales_project_streaming.gld.daily_regional_profit_agg;

In [0]:
data = [
    ('{"name": "Sai", "hobbies":["gym", "code"]}',)
    ]

schema = ("name", "hobby")
jschema = ["value"]

df = spark.createDataFrame(data, jschema)

display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType(), True),
    StructField("hobbies", ArrayType(StringType()), True)
])

df = df.withColumn("parsed", F.from_json("value", schema)).select("parsed.name", "parsed.hobbies")

display(df)

In [0]:
df = df.withColumn("hobby", F.explode_outer("hobbies"))

display(df)

In [0]:
%sql
select * from sales_project_streaming.gld.hourly_regional_profit_agg
where date>current_date()-2;

In [0]:
%sql
select * from sales_project_streaming.gld.weekly_regional_summary_agg

In [0]:
%sql
drop table  sales_project_streaming.brz.dim_employees_historical;

In [0]:
%sql
select * from sales_project_streaming.brz.dim_employees_historical

In [0]:
%sql
describe history sales_project_streaming.brz.dim_employees_historical;

In [0]:
from pyspark.sql import functions as F

df = spark.sql("select window_start from sales_project_streaming.gld.daily_regional_profit_agg")

df = df.withColumn("week_start", F.date_sub(F.date_trunc("week", F.col("window_start")+F.expr("INTERVAL 1 DAY")),1))

display(df)

In [0]:
%sql
select * from sales_project_streaming.gld.weekly_regional_summary_agg;

In [0]:
# %sql
# ---create catalog cdf_sim;
# create schema cdf_sim.brz;
# create schema cdf_sim.slv;
# create schema cdf_sim.gld;

In [0]:
# %sql
# create table cdf_sim.brz.incoming_table(
#     sale_id int,
#     product_id int,
#     amount bigint,
#     sale_date date,
#     ingestion_time timestamp
# )
# using delta;

In [0]:
# %sql
# create table cdf_sim.slv.cdf_sales(
#     sale_id int,
#     product_id int,
#     amount bigint,
#     sale_date date,
#     ingestion_time timestamp
# )
# using delta
# partitioned by (sale_date)
# tblproperties (delta.enableChangeDataFeed = True);

In [0]:
%sql
insert into cdf_sim.brz.incoming_table
values( 5, 20, 7000, "2026-05-10", current_timestamp()),
        ( 10, 25, 9000, "2026-05-10", current_timestamp())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

last_processed_time = spark.sql("""select last_processed_ts
                                from cdf_sim.slv.control_table where topic_name = "sales";""").collect()[0]["last_processed_ts"]

df = spark.read.table("cdf_sim.brz.incoming_table").filter(F.col("ingestion_time")>last_processed_time)

w = Window.partitionBy("sale_id", "sale_date").orderBy(F.col("ingestion_time").desc())
df = df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn")==1).drop("rn")

silver = DeltaTable.forName(spark, "cdf_sim.slv.cdf_sales")

silver.alias("t").merge(
    source = df.alias("s"),
    condition = "s.sale_id = t.sale_id and s.sale_date = t.sale_date"
).whenMatchedUpdate(set = {
    "amount": "s.amount",
    "product_id": "s.product_id",
    "ingestion_time": "s.ingestion_time"
}).whenNotMatchedInsert(values = {
    "sale_id": "s.sale_id",
    "sale_date": "s.sale_date",
    "amount": "s.amount",
    "product_id": "s.product_id",
    "ingestion_time": "s.ingestion_time"
}).execute()

display(df)

In [0]:
last_version = spark.sql("""select last_version from cdf_sim.slv.control_table where topic_name = "sales";""").collect()[0]["last_version"]

print(last_version)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cdf_df = (spark.read.format("delta")
          .option("readChangeFeed", "true")
          .option("startingVersion", last_version+1)
          .table("cdf_sim.slv.cdf_sales"))

cdf_df.display()

meta_df = cdf_df.agg(F.max("ingestion_time").alias("last_processed_ts"), F.max("_commit_version").alias("last_version")).withColumn("topic_name", F.lit("sales"))

control = DeltaTable.forName(spark, "cdf_sim.slv.control_table")

control.alias("t").merge(
    source = meta_df.alias("m"),
    condition = "m.topic_name = t.topic_name"
).whenMatchedUpdate(
    condition = """m.last_processed_ts is not null AND (t.last_processed_ts is null or m.last_processed_ts>t.last_processed_ts)""",
    set = {
        "topic_name": "m.topic_name",
        "last_version": "m.last_version",
        "last_processed_ts": "m.last_processed_ts"
    }
).whenNotMatchedInsertAll().execute()

display(meta_df)

In [0]:
df = cdf_df.groupBy("sale_date").agg(F.sum(F.when(F.col("_change_type")=="update_preimage", -F.col("amount"))
                                       .when(F.col("_change_type")=="update_postimage", F.col("amount"))
                                       .when(F.col("_change_type")=="insert", F.col("amount"))).alias("total_sale"))\
                                           .select("sale_date", "total_sale")

df.display()

In [0]:
%sql
select * from cdf_sim.slv.control_table

In [0]:
# %sql
# create table cdf_sim.slv.control_table(
#     topic_name string,
#     last_version int,
#     last_processed_ts timestamp
# )
# using delta;